# Project 8 - Customer Segmentation

## Objective

Perform customer segmentation using unsupervised machine learning to discover groups of similar customers and generate business recommendations.

### Algorithms
- K-Means Clustering
- Hierarchical Clustering
- PCA Visualization


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv("cleaned_data/telco_cleaned.csv")
df.head()

## Data Preparation

In [ ]:
data = df.copy()

for col in data.select_dtypes(include="object").columns:
    if col != "Churn":
        data[col] = LabelEncoder().fit_transform(data[col].astype(str))

if "Churn" in data.columns:
    data = data.drop(columns=["Churn"])

scaler = StandardScaler()
X = scaler.fit_transform(data)

print("Prepared Shape:", X.shape)

## Elbow Method

In [ ]:
inertia = []

for k in range(2,11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)
    inertia.append(model.inertia_)

plt.figure(figsize=(6,4))
plt.plot(range(2,11), inertia, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

## K-Means Clustering

In [ ]:
k = 4

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

data["Cluster"] = clusters

score = silhouette_score(X, clusters)

print("Silhouette Score:", score)
data["Cluster"].value_counts()

## PCA Visualization

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Customer Segments (PCA)")
plt.show()

## Hierarchical Clustering

In [ ]:
hc = AgglomerativeClustering(n_clusters=k)
hc_labels = hc.fit_predict(X)

print(pd.Series(hc_labels).value_counts())

## Cluster Summary

In [ ]:
summary = data.groupby("Cluster").mean(numeric_only=True)
summary

## Business Recommendations

After reviewing the cluster summary, identify:

- High-value customers
- Loyal customers
- High-risk customers
- Premium plan candidates
- Discount campaign targets


In [ ]:
import os
os.makedirs("segmentation_output", exist_ok=True)
data.to_csv("segmentation_output/customer_segments.csv", index=False)
print("Saved segmentation_output/customer_segments.csv")

# Conclusion

Customer segmentation groups similar customers into meaningful clusters, enabling personalized marketing, retention campaigns, and targeted business strategies.
